# Stage 26: Confirmatory Dose-Response (Corrected)

Stage 25 provided strong exploratory evidence that the magnitude of geometric disruption directly controls the late-stage optimization trajectory of the PINN, producing a graded deterioration in final error. 

In Stage 26A, we discovered an incredible phenomenon: the full SOAP update is hyper-sensitive to $O(10^{-7})$ numerical discrepancies in the reconstructed natural gradient (amplifying them to $O(10^{-2})$ update changes). To prevent this $10^5\times$ empirical local update sensitivity ($\kappa_{\text{emp}}$) from contaminating the reference condition, this confirmatory script forces $\rho=0$ to exactly bypass all intervention logic.

This notebook conducts a definitive, confirmatory run:
1. We test $K \in \{10, 20\}$.
2. We rigorously scale to `SEEDS = [0, 1, 2, 3, 4]`. 
3. We employ $\rho \in \{0.0, 0.25, 0.50, 0.75, 1.0\}$.
4. We record $E_g$, $E_{\text{update}}$, and $\kappa_{\text{emp}}$ globally to track the exact numerical chain of causation.

In [ ]:
import json
import math
import sys
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

sys.path.append('.')
from official_soap import SOAP


In [ ]:
class SOAPStage26(SOAP):
    def __init__(self, params, rho=0.0, base_seed=0, lr=1e-3, betas=(0.95, 0.95), shampoo_beta=-1, 
                 eps=1e-8, weight_decay=0.0, precondition_frequency=10, 
                 max_precond_dim=10000, merge_dims=False, precondition_1d=False,
                 normalize_grads=False, data_format='channels_first', correct_bias=True,
                 mu=0.5):
        super().__init__(params, lr=lr, betas=betas, shampoo_beta=shampoo_beta, 
                         eps=eps, weight_decay=weight_decay, 
                         precondition_frequency=precondition_frequency,
                         max_precond_dim=max_precond_dim, merge_dims=merge_dims, 
                         precondition_1d=precondition_1d, normalize_grads=normalize_grads, 
                         data_format=data_format, correct_bias=correct_bias)
        self.rho = rho
        self.base_seed = base_seed
        self.mu = mu
        self.diagnostics = {}

    def functional_soap_update(self, raw_gradient, state, group, p):
        b1, b2 = group['betas']
        eps = group['eps']
        
        proj_grad = self.project(raw_gradient.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        m_old = state.get("exp_avg", torch.zeros_like(proj_grad))
        v_old = state.get("exp_avg_sq", torch.zeros_like(proj_grad))
        
        m_new = m_old * b1 + proj_grad * (1.0 - b1)
        v_new = v_old * b2 + proj_grad.square() * (1.0 - b2)
        
        denom = v_new.sqrt() + eps
        
        step_val = state.get('step', 0) + 1
        step_size = group['lr']
        if getattr(self, "correct_bias", True):
            bias_correction1 = 1.0 - b1 ** step_val
            bias_correction2 = 1.0 - b2 ** step_val
            step_size = step_size * (math.sqrt(bias_correction2)) / bias_correction1
            
        norm_grad = self.project_back(m_new / denom, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        if group.get("normalize_grads", False):
            norm_grad = norm_grad / (1e-30 + torch.mean(norm_grad**2)**0.5)
            
        update = -step_size * norm_grad
        
        if group.get("weight_decay", 0.0) > 0.0:
            update = update - group['lr'] * group['weight_decay'] * p.data
            
        return update

    @torch.no_grad()
    def step(self, closure=None):
        loss = None if closure is None else closure()
        self.diagnostics = {"norm_update": 0.0, "E_update": 0.0, "E_g": 0.0, "kappa_emp": 0.0, "cos": 0.0, "count": 0}
        
        total_update_sq = 0.0
        total_dnat_sq = 0.0
        total_dint_sq = 0.0
        total_dot = 0.0
        
        total_gdiff_sq = 0.0
        total_gnat_sq = 0.0
        
        current_step = 0
        if len(self.param_groups) > 0 and len(self.param_groups[0]['params']) > 0:
            p_sample = self.param_groups[0]['params'][0]
            if 'step' in self.state[p_sample]:
                current_step = self.state[p_sample]['step'] + 1
        else:
            current_step = 1
        
        idx = 0
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    idx += 1
                    continue
                
                layer_idx = idx
                state = self.state[p]
                raw_g = p.grad.clone()
                
                if 'moq_initialized' not in state:
                    state['moq_initialized'] = True
                    state['prev_raw_g'] = raw_g.clone()
                    g_final = raw_g.clone()
                else:
                    g_nat = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                    
                    if self.rho == 0.0:
                        g_final = g_nat.clone()
                        
                        d_nat = self.functional_soap_update(g_nat, state, group, p)
                        d_int = d_nat.clone()
                        
                        total_dnat_sq += float(torch.sum(d_nat**2))
                        total_dint_sq += float(torch.sum(d_int**2))
                        total_dot += float(torch.sum(d_nat * d_int))
                        
                        total_gnat_sq += float(torch.sum(g_nat**2))
                        self.diagnostics["count"] += 1
                    else:
                        proj_raw = self.project(raw_g.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                        b1, b2 = group["betas"]
                        v_old = state.get("exp_avg_sq")
                        
                        if v_old is not None and p.dim() > 1 and 'Q' in state and state['Q'] is not None:
                            v_raw = v_old * b2 + proj_raw.square() * (1.0 - b2)
                            denom_raw = v_raw.sqrt() + group["eps"]
                            D_inv = 1.0 / denom_raw
                            
                            delta_g_orig = self.mu * (raw_g - state['prev_raw_g'])
                            proj_delta = self.project(delta_g_orig.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                            
                            delta_m = proj_delta * (1.0 - b1)
                            delta_m_norm = float(torch.linalg.vector_norm(delta_m))
                            
                            if delta_m_norm > 1e-12:
                                denom_flat = denom_raw.flatten()
                                k10 = max(1, int(0.10 * denom_flat.numel()))
                                _, indices10 = torch.topk(denom_flat, k10, largest=False)
                                
                                tail_mask = torch.zeros_like(denom_flat, dtype=torch.bool)
                                tail_mask[indices10] = True
                                
                                z_delta = D_inv * delta_m
                                z_delta_sq = z_delta.flatten()**2
                                sum_z_delta_sq = float(torch.sum(z_delta_sq)) + 1e-12
                                F10_nat = float(torch.sum(z_delta_sq[indices10])) / sum_z_delta_sq
                                
                                delta_m_flat = delta_m.flatten()
                                d_T_vals = delta_m_flat[tail_mask]
                                d_N_vals = delta_m_flat[~tail_mask]
                                
                                if len(d_T_vals) > 0 and len(d_N_vals) > 0:
                                    
                                    seed_val = self.base_seed + current_step * 1000 + layer_idx
                                    gen = torch.Generator(device='cpu')
                                    gen.manual_seed(seed_val)
                                    
                                    perm_T = torch.randperm(len(d_T_vals), generator=gen, device='cpu').to(d_T_vals.device)
                                    perm_N = torch.randperm(len(d_N_vals), generator=gen, device='cpu').to(d_N_vals.device)
                                    
                                    d_T_perm = d_T_vals[perm_T]
                                    d_N_perm = d_N_vals[perm_N]
                                    
                                    d_T_mix = (1.0 - self.rho) * d_T_vals + self.rho * d_T_perm
                                    d_N_mix = (1.0 - self.rho) * d_N_vals + self.rho * d_N_perm
                                    
                                    d_perm_flat = torch.zeros_like(delta_m_flat)
                                    d_perm_flat[tail_mask] = d_T_mix
                                    d_perm_flat[~tail_mask] = d_N_mix
                                    d_perm = d_perm_flat.view(delta_m.shape)
                                    
                                    tail_mask_shaped = tail_mask.view(denom_raw.shape)
                                    d_T_full = torch.where(tail_mask_shaped, d_perm, torch.zeros_like(d_perm))
                                    d_N_full = torch.where(~tail_mask_shaped, d_perm, torch.zeros_like(d_perm))
                                    
                                    norm_d_T_sq = float(torch.sum(d_T_full**2))
                                    norm_d_N_sq = float(torch.sum(d_N_full**2))
                                    
                                    E_T = float(torch.sum((D_inv * d_T_full)**2))
                                    E_N = float(torch.sum((D_inv * d_N_full)**2))
                                    
                                    if F10_nat == 0.0:
                                        c_T = 0.0
                                        c_N = 1.0 / math.sqrt(norm_d_N_sq + 1e-12)
                                    elif F10_nat == 1.0:
                                        c_T = 1.0 / math.sqrt(norm_d_T_sq + 1e-12)
                                        c_N = 0.0
                                    else:
                                        ratio = (F10_nat * E_N) / ((1.0 - F10_nat) * E_T + 1e-12)
                                        c_N_sq = 1.0 / (ratio * norm_d_T_sq + norm_d_N_sq + 1e-12)
                                        c_T_sq = c_N_sq * ratio
                                        c_N = math.sqrt(c_N_sq)
                                        c_T = math.sqrt(c_T_sq)
                                        
                                    d_corrected = (c_T * d_T_full + c_N * d_N_full) * delta_m_norm
                                    proj_delta_corrected = d_corrected / (1.0 - b1)
                                    delta_g_corrected = self.project_back(proj_delta_corrected, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                                    g_int = raw_g + delta_g_corrected
                                    g_final = g_int
                                    
                                    d_nat = self.functional_soap_update(g_nat, state, group, p)
                                    d_int = self.functional_soap_update(g_int, state, group, p)
                                    
                                    total_dnat_sq += float(torch.sum(d_nat**2))
                                    total_dint_sq += float(torch.sum(d_int**2))
                                    total_dot += float(torch.sum(d_nat * d_int))
                                    total_update_sq += float(torch.sum((d_int - d_nat)**2))
                                    
                                    total_gdiff_sq += float(torch.sum((g_int - g_nat)**2))
                                    total_gnat_sq += float(torch.sum(g_nat**2))
                                    
                                    self.diagnostics["count"] += 1
                                else:
                                    g_final = g_nat.clone()
                            else:
                                g_final = g_nat.clone()
                        else:
                            g_final = g_nat.clone()
                            
                    state['prev_raw_g'].copy_(raw_g)
                
                p.grad.copy_(g_final)
                idx += 1
                
        old_params = {}
        for group in self.param_groups:
            for p in group['params']:
                old_params[p] = p.data.clone()
                
        super().step()
        
        actual_update_sq = 0.0
        for group in self.param_groups:
            for p in group['params']:
                actual_update_sq += float(torch.sum((p.data - old_params[p])**2))
                
        self.diagnostics["norm_update"] = math.sqrt(actual_update_sq)
        if self.diagnostics["count"] > 0:
            norm_dnat = math.sqrt(total_dnat_sq)
            norm_dint = math.sqrt(total_dint_sq)
            
            E_up = math.sqrt(total_update_sq) / (norm_dnat + 1e-12)
            self.diagnostics["E_update"] = E_up
            self.diagnostics["cos"] = total_dot / (norm_dnat * norm_dint + 1e-12)
            
            if self.rho > 0.0:
                E_g = math.sqrt(total_gdiff_sq) / (math.sqrt(total_gnat_sq) + 1e-12)
                self.diagnostics["E_g"] = E_g
                self.diagnostics["kappa_emp"] = E_up / (E_g + 1e-12)
            
        return loss


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None
    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule

def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def run_stage26_training(rho: float, K: int=1, steps: int=2000, seed: int=0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule_long.pt")
    if schedule_path.exists():
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
        if len(schedule) < steps:
            schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
            torch.save(schedule, schedule_path)
    else:
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    opt = SOAPStage26(
        model.parameters(), 
        rho=rho,
        base_seed=seed,
        lr=1e-3, 
        betas=(0.95, 0.95), 
        precondition_frequency=10, 
        mu=0.5
    )

    history = []
    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        opt.step()
        
        if step % 10 == 0 or step == 1:
            err = relative_l2(model, device)
            rec = {
                "step": step,
                "loss_pde": loss_pde.item(),
                "rel_l2": err,
                "norm_update": opt.diagnostics.get("norm_update", 0.0),
                "E_update": opt.diagnostics.get("E_update", 0.0),
                "E_g": opt.diagnostics.get("E_g", 0.0),
                "kappa_emp": opt.diagnostics.get("kappa_emp", 0.0),
                "cos": opt.diagnostics.get("cos", 0.0)
            }
            history.append(rec)
            
            if not math.isfinite(loss.item()) or not math.isfinite(err):
                print(f"Divergence detected at step {step} for rho {rho}")
                break
                
    best_rel_l2 = min([r["rel_l2"] for r in history])
    step_of_best = next(r["step"] for r in history if r["rel_l2"] == best_rel_l2)
    e_up = [r["E_update"] for r in history if r["step"] > 1]
    
    summary = {
        "final_rel_l2": history[-1]["rel_l2"],
        "best_rel_l2": best_rel_l2,
        "step_of_best": step_of_best,
        "final_pde_loss": history[-1]["loss_pde"],
        "mean_E_update": float(np.mean(e_up)) if e_up else 0.0,
        "median_E_update": float(np.median(e_up)) if e_up else 0.0,
        "p90_E_update": float(np.percentile(e_up, 90)) if e_up else 0.0,
        "max_E_update": float(np.max(e_up)) if e_up else 0.0
    }

    return summary, history


### Stage 26: Execute Confirmatory Training and Visualization

In [ ]:
import matplotlib.pyplot as plt

rhos = [0.0, 0.25, 0.5, 0.75, 1.0]
Ks = [10, 20]
seeds = [0, 1, 2, 3, 4]

all_results = {}

for K in Ks:
    all_results[K] = {}
    for rho in rhos:
        all_results[K][rho] = {"summaries": {}, "trajectories": {}}
        for seed in seeds:
            print(f"\n{'='*40}\nRunning K={K} Rho={rho} Seed={seed}...\n{'='*40}")
            summary, history = run_stage26_training(rho=rho, K=K, seed=seed, steps=2000)
            all_results[K][rho]["summaries"][seed] = summary
            all_results[K][rho]["trajectories"][seed] = history
            
            if rho == 0.0:
                print(f"-> Max E_update at rho=0: {summary['max_E_update']:.4e} (Must be 0.0)")
            
    with open(f"stage26_K{K}_confirmatory.json", "w") as f:
        json.dump(all_results[K], f, indent=2)

    # --- Figure 1: Dose-Response (rho vs final RelL2) ---
    mean_rels = []
    std_rels = []
    for rho in rhos:
        finals = [all_results[K][rho]["summaries"][s]["final_rel_l2"] for s in seeds]
        mean_rels.append(np.mean(finals))
        std_rels.append(np.std(finals))
    
    plt.figure(figsize=(8, 6))
    plt.errorbar(rhos, mean_rels, yerr=std_rels, fmt='-o', capsize=5, label=f'K={K}')
    plt.xlabel('Intervention Strength (rho)')
    plt.ylabel('Final RelL2')
    plt.title(f'K={K}: Final Error vs Geometric Disruption')
    plt.grid(True)
    plt.legend()
    plt.savefig(f'stage26_K{K}_dose_response.png')
    plt.close()

    # --- Figure 2: Trajectories (mean RelL2 vs step) ---
    plt.figure(figsize=(10, 6))
    steps_t = [r["step"] for r in all_results[K][rhos[0]]["trajectories"][seeds[0]]]
    for rho in rhos:
        all_traj = []
        for s in seeds:
            all_traj.append([r["rel_l2"] for r in all_results[K][rho]["trajectories"][s]])
        mean_traj = np.mean(all_traj, axis=0)
        plt.plot(steps_t, mean_traj, label=f'rho={rho}')
    plt.yscale('log')
    plt.xlabel('Step')
    plt.ylabel('RelL2 (log scale)')
    plt.title(f'K={K}: Training Trajectories by Rho')
    plt.grid(True, which="both", ls="-")
    plt.legend()
    plt.savefig(f'stage26_K{K}_trajectories.png')
    plt.close()

    # --- Figure 3: Mechanism -> Outcome (Mean E_update vs Final RelL2) ---
    plt.figure(figsize=(8, 6))
    for rho in rhos:
        e_ups = [all_results[K][rho]["summaries"][s]["mean_E_update"] for s in seeds]
        finals = [all_results[K][rho]["summaries"][s]["final_rel_l2"] for s in seeds]
        plt.scatter(e_ups, finals, label=f'rho={rho}', alpha=0.7)
    plt.xlabel('Mean E_update (Trajectory Average)')
    plt.ylabel('Final RelL2')
    plt.title(f'K={K}: Local Perturbation vs Global Error')
    plt.grid(True)
    plt.legend()
    plt.savefig(f'stage26_K{K}_mechanism_outcome.png')
    plt.close()
